In [21]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = "2"
import tensorflow as tf
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
tf.random.set_seed(22)
np.random.seed(22)
assert tf.__version__.startswith('2.')

batch_size = 128
total_words = 10000
max_review_len = 80
embedding_len = 100
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.imdb.load_data(num_words=total_words)

x_train = tf.keras.preprocessing.sequence.pad_sequences(x_train,
maxlen=max_review_len)
x_test = tf.keras.preprocessing.sequence.pad_sequences(x_test,
maxlen=max_review_len)

train_data = tf.data.Dataset.from_tensor_slices((x_train, y_train))
train_data = train_data.shuffle(10000).batch(batch_size,
drop_remainder=True)
test_data = tf.data.Dataset.from_tensor_slices((x_test, y_test))
test_data = test_data.batch(batch_size, drop_remainder=True)
print('x_train_shape:', x_train.shape, tf.reduce_max(y_train),
tf.reduce_min(y_train))
print('x_test_shape:', x_test.shape)

sample = next(iter(test_data))
print(sample[0].shape)
class RNN_Build(tf.keras.Model):
  def __init__(self, units):
    super(RNN_Build, self).__init__()
    self.embedding = tf.keras.layers.Embedding(total_words, embedding_len)
    # Refactor to use SimpleRNN layers for proper tf.function tracing and dropout handling
    self.rnn = tf.keras.Sequential([
      tf.keras.layers.SimpleRNN(units, dropout=0.2, return_sequences=True),
      tf.keras.layers.SimpleRNN(units, dropout=0.2)
    ])
    self.outlayer = tf.keras.layers.Dense(1)
  def call(self, inputs, training=None):
    x = inputs
    x = self.embedding(x)
    x = self.rnn(x, training=training) # Pass training argument to the Sequential RNN block
    x = self.outlayer(x)
    prob = tf.sigmoid(x)
    return prob
import time
units = 64
epochs = 4
t0 = time.time()

model = RNN_Build(units)
model.compile(optimizer=tf.keras.optimizers.Adam(0.001),
  loss=tf.losses.BinaryCrossentropy(),
  metrics=['accuracy'])

model.fit(train_data, epochs=epochs, validation_data=test_data,
validation_freq=2)
print("훈련 데이터셋 평가...")
(loss, accuracy) = model.evaluate(train_data, verbose=0)
print("loss={:.4f}, accuracy: {:.4f}%".format(loss,accuracy *
100))
print("테스트 데이터셋 평가...")
(loss, accuracy) = model.evaluate(test_data, verbose=0)
print("loss={:.4f}, accuracy: {:.4f}%".format(loss,accuracy *
100))
t1 = time.time()
print('시간:', t1-t0)

x_train_shape: (25000, 80) tf.Tensor(1, shape=(), dtype=int64) tf.Tensor(0, shape=(), dtype=int64)
x_test_shape: (25000, 80)
(128, 80)
Epoch 1/4
195/195 ━━━━━━━━━━━━━━━━━━━━ 21s 90ms/step - accuracy: 0.5831 - loss: 0.6569
Epoch 2/4
195/195 ━━━━━━━━━━━━━━━━━━━━ 24s 108ms/step - accuracy: 0.8274 - loss: 0.3923 - val_accuracy: 0.8200 - val_loss: 0.4290
Epoch 3/4
195/195 ━━━━━━━━━━━━━━━━━━━━ 34s 75ms/step - accuracy: 0.9123 - loss: 0.2277
Epoch 4/4
195/195 ━━━━━━━━━━━━━━━━━━━━ 20s 100ms/step - accuracy: 0.9631 - loss: 0.1033 - val_accuracy: 0.7866 - val_loss: 0.6553
훈련 데이터셋 평가...
loss=0.0323, accuracy: 99.1987%
테스트 데이터셋 평가...
loss=0.6553, accuracy: 78.6579%
시간: 108.21025013923645


In [18]:
class RNN_Build(tf.keras.Model):
  def __init__(self, units):
    super(RNN_Build, self).__init__()
    self.embedding = tf.keras.layers.Embedding(total_words, embedding_len, input_length=max_review_len)
    self.rnn = tf.keras.Sequential([
      tf.keras.layers.SimpleRNN(units, dropout=0.5,
  return_sequences=True),
      tf.keras.layers.SimpleRNN(units, dropout=0.5)
    ])
    self.outlayer = tf.keras.layers.Dense(1)
  def call(self, inputs, training=None):
    x = inputs
    x = self.embedding(x)
    x = self.rnn(x)
    x = self.outlayer(x)
    prob = tf.sigmoid(x)

    return prob
import time
units = 64
epochs = 4
t0 = time.time()

model = RNN_Build(units)

model.compile(optimizer=tf.keras.optimizers.Adam(0.001),
  loss=tf.losses.BinaryCrossentropy(),
  metrics=['accuracy'])

model.fit(train_data, epochs=epochs, validation_data=test_data,
validation_freq=2)
print("훈련 데이터셋 평가...")
(loss, accuracy) = model.evaluate(train_data, verbose=0)
print("loss={:.4f}, accuracy: {:.4f}%".format(loss,accuracy * 100))
print("테스트 데이터셋 평가...")
(loss, accuracy) = model.evaluate(test_data, verbose=0)
print("loss={:.4f}, accuracy: {:.4f}%".format(loss,accuracy * 100))

t1 = time.time()
print('시간:', t1-t0)

Epoch 1/4


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


195/195 ━━━━━━━━━━━━━━━━━━━━ 22s 99ms/step - accuracy: 0.5286 - loss: 0.6937
Epoch 2/4
195/195 ━━━━━━━━━━━━━━━━━━━━ 21s 104ms/step - accuracy: 0.7663 - loss: 0.4888 - val_accuracy: 0.8101 - val_loss: 0.4309
Epoch 3/4
195/195 ━━━━━━━━━━━━━━━━━━━━ 17s 87ms/step - accuracy: 0.8595 - loss: 0.3384
Epoch 4/4
195/195 ━━━━━━━━━━━━━━━━━━━━ 24s 106ms/step - accuracy: 0.9034 - loss: 0.2438 - val_accuracy: 0.8165 - val_loss: 0.4842
훈련 데이터셋 평가...
loss=0.1029, accuracy: 96.4543%
테스트 데이터셋 평가...
loss=0.4842, accuracy: 81.6546%
시간: 91.84884262084961
